# Airbnb Cape Town Capstone (Starter Notebook)

**Student:** Precious  
**Dataset:** Inside Airbnb, Cape Town listings snapshot (2026-09-23), CC BY 4.0  

**Two tasks, one dataset (clean/EDA/split shared):**
- **Task C (classification, deployed):** predict `host_is_superhost`
- **Task R (regression, notebook only):** predict `price`

> This is a **starter/sample notebook**. You must understand, edit, and be able to explain every cell out loud.  
> Replace all `TODO` items with your own work and add markdown findings after important results.

## A–B. Setup, framing, and targets (Checklist A, B)

**TODO:**
- Who uses each prediction and what decision it supports?
- Why is Task C classification and Task R regression?
- Define targets: `host_is_superhost` → 0/1 (print class counts). `price` → numeric (state what you do with missing/invalid prices).
- Choose metrics **before** modelling and justify them (Task C: why not accuracy alone; Task R: RMSE vs MAE).

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import (
    GroupShuffleSplit, GroupKFold, cross_validate, cross_val_predict,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    mean_squared_error, mean_absolute_error, silhouette_score
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available:', e)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Everything works')
print('pandas', pd.__version__)
print('numpy', np.__version__)
import sklearn
print('sklearn', sklearn.__version__)
print('xgboost available:', XGBOOST_AVAILABLE)

Everything works
pandas 2.3.3
numpy 2.0.2
sklearn 1.6.1
xgboost available: True


In [2]:
PROJECT_DIR = Path('..').resolve()
DATA_DIR = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
MODELS_DIR = PROJECT_DIR / 'models'
FIG_DIR = PROJECT_DIR / 'reports' / 'figures'

for p in [PROCESSED_DIR, MODELS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LISTINGS_PATH = DATA_DIR / 'listings.csv.gz'
NEIGH_PATH = DATA_DIR / 'neighbourhoods.csv'

print('Project:', PROJECT_DIR)
print('Listings exists:', LISTINGS_PATH.exists(), '-', LISTINGS_PATH)
print('Neighbourhoods exists:', NEIGH_PATH.exists(), '-', NEIGH_PATH)
print('Processed dir:', PROCESSED_DIR)
print('Models dir:', MODELS_DIR)


Project: C:\Users\Chibi\Desktop\New folder\Workspace\Project\CAMLDS\Final\Final Capstone\airbnb-cape-town
Listings exists: True - C:\Users\Chibi\Desktop\New folder\Workspace\Project\CAMLDS\Final\Final Capstone\airbnb-cape-town\data\raw\listings.csv.gz
Neighbourhoods exists: True - C:\Users\Chibi\Desktop\New folder\Workspace\Project\CAMLDS\Final\Final Capstone\airbnb-cape-town\data\raw\neighbourhoods.csv
Processed dir: C:\Users\Chibi\Desktop\New folder\Workspace\Project\CAMLDS\Final\Final Capstone\airbnb-cape-town\data\processed
Models dir: C:\Users\Chibi\Desktop\New folder\Workspace\Project\CAMLDS\Final\Final Capstone\airbnb-cape-town\models


## B. Problem framing (Checklist B)

### B1. Business questions

*What is a superhost?* On Airbnb, a Superhost is a host recognized for consistently providing strong guest experiences. It’s a status, just like platinum driver. 
**Task C - Superhost classification (deployed):**  
*Who uses it?* An Airbnb support or host-success team in Cape Town.  
*What decision does it support?* The model estimates whether a listing's host profile looks like a Superhost. The team can use this as a prioritization signal: for example, flag listings/hosts that look close to Superhost quality for coaching, or review cases where the model disagrees with expectations. It is not an official Airbnb label and should not replace Airbnb's own rules.

**Task R - Price regression (notebook only):**  
*Who uses it?* A host, listing manager, or analyst.  
*What decision does it support?* The model estimates a reasonable nightly price range based on listing characteristics. It can help a host sanity-check pricing, spot under/over-priced listings, or understand which features are associated with higher prices.

**Why classification vs regression?**  
Task C is classification because `host_is_superhost` is a category/label (`t` or `f`) - the answer is "which class?". Task R is regression because `price` is a numeric amount in rand - the answer is "how much?".

### B2. Target definitions
- `host_is_superhost`: map `t -> 1`, `f -> 0`. I print class counts after loading/cleaning. Rows with missing target are excluded from Task C modelling.
- `price`: parse currency strings like `$1,114.00` into numeric rand values. Missing/blank/invalid prices become `NaN` and are excluded from Task R modelling (I also report how many rows are affected). I inspect extreme prices before deciding whether to cap/remove them.

### B3. Metrics chosen before modelling
**Task C (Superhost, class imbalance expected):** I will report **F1, precision, recall, accuracy, and ROC-AUC**.  
Why not accuracy alone? Because if most hosts are not Superhosts, a model that predicts "not Superhost" for everyone can get decent accuracy while catching zero actual Superhosts. That would be useless for the support team. Recall matters when missing real Superhosts is costly; precision matters when false alarms waste team time. F1 balances precision and recall, and ROC-AUC summarizes ranking quality across thresholds.

**Task R (price):** I will report **RMSE and MAE** in rand.  
RMSE squares errors before averaging, so it punishes large mistakes more heavily. MAE is the average absolute error and is easier to read as "typical rand error". I use both because a model might have acceptable average error (MAE) but still make some very bad large-price mistakes (RMSE exposes this).

## C. Data loading and inspection (Checklist C)

Tasks: load raw gz, print shape (expect ~27760 x 90), inspect dtypes/missing/value counts, fix `price` and `bathrooms_text`, audit missing/empty cols, duplicates and host repeats, merge `neighbourhoods.csv` and verify row counts.

In [ ]:
# C1 - Load raw listings
# compression='gzip' lets pandas read the .gz directly without manually unzipping.
df_raw = pd.read_csv(LISTINGS_PATH, compression='gzip', low_memory=False)
print('Shape:', df_raw.shape)
print('Rows:', df_raw.shape[0], 'Columns:', df_raw.shape[1])
df_raw.head(3)


In [ ]:
# C2 - Basic inspection
print('--- info ---')
df_raw.info()

print('--- numeric describe ---')
display(df_raw.describe().T.head(20))

# Key categoricals / targets
for col in ['host_is_superhost', 'room_type', 'neighbourhood_cleansed', 'instant_bookable', 'property_type']:
    if col in df_raw.columns:
        print(f'--- value_counts: {col} ---')
        print(df_raw[col].value_counts(dropna=False).head(10))


In [ ]:
# C3 - Dtype fixes: price and bathrooms
df = df_raw.copy()

# Price: '$1,114.00' -> 1114.00; blanks/invalid -> NaN
price_before = df['price'].head(10).copy()
df['price_clean'] = (
    df['price'].astype('string')
      .str.replace('$', '', regex=False)
      .str.replace(',', '', regex=False)
      .str.strip()
)
df['price_clean'] = pd.to_numeric(df['price_clean'].replace({'': np.nan, 'nan': np.nan}), errors='coerce')

# Bathrooms: extract numeric value; treat 'half-bath'/'half bath' as 0.5
bath_before = df['bathrooms_text'].dropna().head(12).copy()
bath_text = df['bathrooms_text'].astype('string').str.lower().str.strip()
bath_num = bath_text.str.extract(r'(\d+(?:\.\d+)?)', expand=False)
df['bathrooms_count'] = pd.to_numeric(bath_num, errors='coerce')
half_mask = bath_text.str.contains('half', na=False)
df.loc[half_mask & df['bathrooms_count'].isna(), 'bathrooms_count'] = 0.5
# If text said half-bath but regex captured a number in odd cases, keep explicit 0.5 only when clearly half.
df.loc[bath_text.str.fullmatch(r'half-bath|half bath', na=False), 'bathrooms_count'] = 0.5

print('Price before -> after:')
display(pd.DataFrame({'price_raw': price_before, 'price_clean': df.loc[price_before.index, 'price_clean']}))
print('\nBathrooms before -> after:')
display(pd.DataFrame({'bathrooms_text': bath_before, 'bathrooms_count': df.loc[bath_before.index, 'bathrooms_count']}))
print('price_clean missing:', df['price_clean'].isna().sum())
print('bathrooms_count missing:', df['bathrooms_count'].isna().sum())


In [ ]:
# C4 - Missing value audit
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values(['missing_pct', 'missing_count'], ascending=False)

print('Columns with any missing:', (missing['missing_count'] > 0).sum())
print('Fully empty columns (100% missing):')
display(missing[missing['missing_pct'] == 100])
display(missing.head(30))


In [ ]:
# C5 - Duplicates and host repeats
dup_rows = df.duplicated().sum()
print('Duplicate rows:', dup_rows)

host_counts = df['host_id'].value_counts()
multi_host = (host_counts > 1).sum()
single_host = (host_counts == 1).sum()
print('Unique hosts:', df['host_id'].nunique())
print('Hosts with >1 listing:', multi_host)
print('Hosts with 1 listing:', single_host)
print('\nTop host listing counts:')
display(host_counts.head(10))


In [ ]:
# C6 - Merge neighbourhoods.csv
neigh = pd.read_csv(NEIGH_PATH)
print('Neighbourhoods shape:', neigh.shape)
display(neigh.head())
print('Neighbourhood columns:', list(neigh.columns))

# listings has neighbourhood_cleansed; neighbourhoods.csv has neighbourhood (and neighbourhood_group)
df_merged = df.merge(
    neigh,
    left_on='neighbourhood_cleansed',
    right_on='neighbourhood',
    how='left',
    suffixes=('', '_neigh')
)

print('Listings shape before merge:', df.shape)
print('Listings shape after merge:', df_merged.shape)
print('Rows lost/duplicated?', df_merged.shape[0] - df.shape[0])
neigh_col = 'neighbourhood_neigh' if 'neighbourhood_neigh' in df_merged.columns else ('neighbourhood_y' if 'neighbourhood_y' in df_merged.columns else None)
print('Unmatched neighbourhoods:', df_merged[neigh_col].isna().sum() if neigh_col else 'n/a')

# Quick check: left join keeps all listing rows; inner join would drop unmatched listings.
df_inner = df.merge(neigh, left_on='neighbourhood_cleansed', right_on='neighbourhood', how='inner')
print('Inner merge shape:', df_inner.shape)


## D. Cleaning (Checklist D)

Do cleaning in code, in order, reproducibly. Remove useless columns with reasons, handle rows/prices/min nights with justification, examine price skew/transform, and create at least one engineered feature (e.g. amenities count) to evaluate later (I5).

In [ ]:
# D1/D2 - Cleaning in one place
clean = df_merged.copy()

# Standardize target
clean['is_superhost'] = clean['host_is_superhost'].map({'t': 1, 'f': 0})

# Parse instant_bookable if present (snapshot may be blank; keep robust)
if 'instant_bookable' in clean.columns:
    clean['instant_bookable_flag'] = clean['instant_bookable'].map({'t': 1, 'f': 0})

# Parse percentage-like host fields if present
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in clean.columns:
        clean[col + '_num'] = pd.to_numeric(
            clean[col].astype('string').str.replace('%', '', regex=False).str.strip(),
            errors='coerce'
        )

# D2 - Drop useless columns by group (only if present)
id_url_cols = [c for c in clean.columns if c in [
    'id','listing_url','scrape_id','last_scraped','source','picture_url',# 'host_id',  # keep for group split; drop later before modeling'host_url','host_name',
    'host_thumbnail_url','host_picture_url','host_neighbourhood','listing_id'
] or c.endswith('_url')]

text_cols = [c for c in ['name','description','neighborhood_overview','host_about','amenities'] if c in clean.columns]
# Keep amenities raw temporarily for D5 feature engineering, then drop it after creating amenities_count.
text_cols_no_amenities = [c for c in text_cols if c != 'amenities']

constant_or_snapshot_cols = [c for c in clean.columns if c.startswith('calendar_') or c in ['calendar_last_scraped']]
fully_empty = [c for c in clean.columns if clean[c].isna().all()]

drop_cols = sorted(set(id_url_cols + text_cols_no_amenities + constant_or_snapshot_cols + fully_empty))
print('Dropping', len(drop_cols), 'columns')
print(drop_cols[:30])
clean = clean.drop(columns=[c for c in drop_cols if c in clean.columns])

print('Shape after initial drops:', clean.shape)


In [ ]:
# D3 - Row handling with reasons
rows_before = len(clean)
print('Rows before:', rows_before)

# 1) Duplicate rows
dup = clean.duplicated().sum()
clean = clean.drop_duplicates().copy()
print('Removed duplicate rows:', dup)

# 2) Target availability
missing_superhost = clean['is_superhost'].isna().sum()
print('Missing is_superhost:', missing_superhost)
# Keep rows; Task C will filter, Task R can still use rows if price is valid.

# 3) Price issues - inspect before removing
missing_price = clean['price_clean'].isna().sum()
print('Missing/invalid price_clean:', missing_price)

# Inspect extremes row by row (small sample) before deciding
price_nonnull = clean.loc[clean['price_clean'].notna(), ['price_clean','minimum_nights','room_type','neighbourhood_cleansed','accommodates']].copy()
display(price_nonnull.sort_values('price_clean').head(10))
display(price_nonnull.sort_values('price_clean', ascending=False).head(10))

# Decision: remove rows with non-positive or missing price for Task R dataset; for shared cleaning keep a flag.
clean['price_valid'] = clean['price_clean'].notna() & (clean['price_clean'] > 0)
print('Invalid/non-positive/missing price rows:', (~clean['price_valid']).sum())

# Minimum nights extremes - inspect before deciding
if 'minimum_nights' in clean.columns:
    mn = pd.to_numeric(clean['minimum_nights'], errors='coerce')
    clean['minimum_nights'] = mn
    display(clean.loc[mn.notna(), ['minimum_nights','price_clean','room_type','neighbourhood_cleansed']].sort_values('minimum_nights', ascending=False).head(10))
    # Decision: keep rows but cap extreme minimum_nights for modelling features later if needed;
    # remove impossible negatives (if any). Here we only remove clearly invalid negatives.
    bad_mn = clean['minimum_nights'] < 0
    print('Negative minimum_nights rows:', bad_mn.sum())
    clean = clean.loc[~bad_mn].copy()

rows_after = len(clean)
print('Rows removed total:', rows_before - rows_after)
print('Rows after:', rows_after)


In [ ]:
# D4 - Price skew and transform
price_valid = clean.loc[clean['price_valid'], 'price_clean'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(price_valid.clip(upper=price_valid.quantile(0.99)), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Price (clipped at 99th pct for view)')
axes[0].set_xlabel('Price (ZAR)')

axes[1].hist(np.log1p(price_valid), bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('log1p(price)')
axes[1].set_xlabel('log1p(price)')
plt.tight_layout()
plt.show()

print('Raw price skew:', price_valid.skew())
print('log1p price skew:', np.log1p(price_valid).skew())

# Inverse transform demo
example_log = np.log1p(price_valid.median())
print('Median price:', price_valid.median())
print('log1p(median):', example_log)
print('Inverse expm1:', np.expm1(example_log))


In [ ]:
# D5 - Engineered feature: amenities_count
if 'amenities' in clean.columns:
    am = clean['amenities'].fillna('[]').astype('string').str.strip()
    # Empty list / missing-like values -> 0
    empty_am = am.isin(['[]', '', 'nan', 'None'])
    # Count commas + 1 for non-empty lists; this is a simple proxy.
    clean['amenities_count'] = am.str.count(',') + 1
    clean.loc[empty_am, 'amenities_count'] = 0
    clean['amenities_count'] = clean['amenities_count'].fillna(0).astype(int)
    print('Amenities count - before/after sample:')
    display(clean.loc[~empty_am, ['amenities', 'amenities_count']].head(5))
    display(clean['amenities_count'].describe())
    # Drop raw amenities text after engineering
    clean = clean.drop(columns=['amenities'])
else:
    clean['amenities_count'] = 0
    print('amenities column not found; created amenities_count=0 placeholder')

# Save cleaned snapshot (raw stays untouched in data/raw)
clean_path = PROCESSED_DIR / 'listings_cleaned.parquet'
try:
    clean.to_parquet(clean_path, index=False)
    print('Saved:', clean_path)
except Exception as e:
    # Fallback if parquet engine unavailable
    csv_path = PROCESSED_DIR / 'listings_cleaned.csv'
    clean.to_csv(csv_path, index=False)
    print('Parquet failed, saved CSV instead:', csv_path)
    print(e)


## E. EDA with written findings (Checklist E)

Include: target plots, ≥4 distributions, ≥4 feature-vs-target plots, at least one `groupby().agg()` table, correlation check, and a **5-point EDA narrative** in your own words (each point tied to evidence).

In [ ]:
# E1 - Target plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Superhost split
target_counts = clean['is_superhost'].value_counts(dropna=False).sort_index()
labels = ['Not Superhost (0)' if i==0 else 'Superhost (1)' if i==1 else 'Missing' for i in target_counts.index]
axes[0].bar(labels, target_counts.values, color=['#4C78A8', '#F58518', '#999999'][:len(labels)])
axes[0].set_title('Superhost class split')
axes[0].set_ylabel('Listings')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v, str(v), ha='center', va='bottom')

# Price histogram
price_valid = clean.loc[clean['price_valid'], 'price_clean']
axes[1].hist(price_valid.clip(upper=price_valid.quantile(0.99)), bins=50, color='teal', edgecolor='white')
axes[1].set_title('Price distribution (clipped at 99th pct)')
axes[1].set_xlabel('Price (ZAR)')
axes[1].set_ylabel('Listings')
plt.tight_layout()
plt.show()

print(clean['is_superhost'].value_counts(normalize=True, dropna=False))
print(price_valid.describe())


In [ ]:
# E2 - At least 4 distribution plots
num_cols = [c for c in ['price_clean','accommodates','bedrooms','bathrooms_count','minimum_nights','number_of_reviews','review_scores_rating','availability_365','amenities_count'] if c in clean.columns]
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols[:9]):
    s = pd.to_numeric(clean[col], errors='coerce').dropna()
    if col in ['price_clean','minimum_nights','number_of_reviews']:
        s = s.clip(upper=s.quantile(0.99))
    axes[i].hist(s, bins=40, color='#59A14F', edgecolor='white')
    axes[i].set_title(col)
for j in range(len(num_cols[:9]), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

# Boxplots for outlier view
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, ['price_clean','minimum_nights','number_of_reviews','availability_365']):
    s = pd.to_numeric(clean[col], errors='coerce').dropna()
    ax.boxplot(s.clip(upper=s.quantile(0.99)), vert=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
# E3 - Feature vs target plots (>=4)
plot_df = clean.copy()
plot_df['price_clean'] = pd.to_numeric(plot_df['price_clean'], errors='coerce')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1) Superhost rate by room_type
rt = plot_df.groupby('room_type')['is_superhost'].mean().sort_values(ascending=False)
axes[0,0].bar(rt.index.astype(str), rt.values, color='#4C78A8')
axes[0,0].set_title('Superhost rate by room type')
axes[0,0].tick_params(axis='x', rotation=20)
axes[0,0].set_ylabel('Mean is_superhost')

# 2) Price by room_type
sns.boxplot(data=plot_df.loc[plot_df['price_valid']], x='room_type', y=plot_df.loc[plot_df['price_valid'],'price_clean'].clip(upper=plot_df.loc[plot_df['price_valid'],'price_clean'].quantile(0.95)), ax=axes[0,1])
axes[0,1].set_title('Price by room type (95th pct clipped)')
axes[0,1].tick_params(axis='x', rotation=20)
axes[0,1].set_ylabel('Price (ZAR)')

# 3) Superhost rate by accommodates bins
plot_df['accom_bin'] = pd.cut(pd.to_numeric(plot_df['accommodates'], errors='coerce'), bins=[0,1,2,4,6,20], include_lowest=True)
ab = plot_df.groupby('accom_bin')['is_superhost'].mean()
axes[1,0].bar(ab.index.astype(str), ab.values, color='#F58518')
axes[1,0].set_title('Superhost rate by accommodates bin')
axes[1,0].tick_params(axis='x', rotation=20)

# 4) Median price by neighbourhood (top 10 by count)
top_neigh = plot_df['neighbourhood_cleansed'].value_counts().head(10).index
np_ = plot_df[plot_df['neighbourhood_cleansed'].isin(top_neigh) & plot_df['price_valid']]
med = np_.groupby('neighbourhood_cleansed')['price_clean'].median().sort_values(ascending=False)
axes[1,1].bar(med.index.astype(str), med.values, color='#54A24B')
axes[1,1].set_title('Median price by top neighbourhoods')
axes[1,1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# E4 - groupby().agg() summary
summary = plot_df.loc[plot_df['price_valid']].groupby('room_type').agg(
    listings=('room_type','size'),
    superhost_rate=('is_superhost','mean'),
    median_price=('price_clean','median'),
    mean_reviews=('number_of_reviews','mean'),
    median_availability=('availability_365','median')
).sort_values('listings', ascending=False)
display(summary.round(3))


In [ ]:
# E5 - Correlation check
corr_cols = [c for c in ['price_clean','accommodates','bedrooms','bathrooms_count','minimum_nights','number_of_reviews','review_scores_rating','reviews_per_month','availability_365','amenities_count','calculated_host_listings_count','is_superhost'] if c in clean.columns]
corr_df = clean[corr_cols].apply(pd.to_numeric, errors='coerce')
corr = corr_df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation matrix (numeric features)')
plt.tight_layout()
plt.show()
display(corr['price_clean'].sort_values(ascending=False).to_frame('corr_with_price').head(12))
display(corr['is_superhost'].sort_values(ascending=False).to_frame('corr_with_superhost').head(12))


## F. Leakage review and split (Checklist F)

**Critical:** write leakage review before modelling (include Airbnb Superhost rules in your own summary).  
Use a **group-aware split** so all listings from the same `host_id` stay in the same set; compare with a plain random split on one model to show why it matters.  
Create train/val/test with fixed seed and report sizes/class rates. Fit preprocessing on train only.

In [ ]:
# F1 - Leakage review
# Airbnb Superhost criteria (summary in my own words):
# A host generally needs to meet performance standards over an evaluation period,
# such as hosting experience/trips or nights, high overall rating, low cancellation rate,
# and high response rate. Exact thresholds can change over time, but these are performance-based.

leakage_review = pd.DataFrame([
    ['host_is_superhost / is_superhost', 'Task C target', 'Remove from features; this is the label.'],
    ['host_response_rate / host_acceptance_rate', 'Potential leakage/proxy', 'Performance-related; may encode Superhost-like behavior. Exclude from Task C.'],
    ['calculated_host_listings_count* / host_listings_count*', 'Borderline host-level', 'Host portfolio info; can leak host identity patterns across rows. I exclude from final app features and be careful in Task C.'],
    ['review_scores_*', 'Borderline', 'Known after reviews exist; useful but possibly proxy for quality. Keep for notebook experiments with caution, but do not treat as causal.'],
    ['price / price_clean', 'Task R target / related', 'Price is Task R target. For Task C, price is known at listing time but I keep it out of the deployed app unless justified; I test carefully.'],
    ['id, host_id, URLs, scrape dates', 'Identifiers/metadata', 'Not predictive features; host_id used only for grouping.'],
    ['amenities_count, room_type, accommodates, bedrooms, bathrooms_count, neighbourhood', 'Generally safe', 'Known listing attributes at prediction time.'],
], columns=['Column/group', 'Assessment', 'Decision/reason'])
display(leakage_review)

# Columns I will explicitly exclude from Task C features
TASK_C_LEAKY_OR_ID = [
    'host_is_superhost','is_superhost','host_id','id','listing_url','scrape_id','last_scraped',
    'host_url','host_name','host_thumbnail_url','host_picture_url',
    'host_response_time','host_response_rate','host_acceptance_rate','host_response_rate_num','host_acceptance_rate_num'
]
print('Example excluded columns:', [c for c in TASK_C_LEAKY_OR_ID if c in clean.columns])


In [ ]:
# F2/F3 - Group-aware split by host_id
# We need host_id present for grouping. If it was dropped, reload or keep it earlier.
if 'host_id' not in clean.columns:
    clean['host_id'] = df_raw['host_id']

model_df = clean.copy()
model_df = model_df[model_df['is_superhost'].notna()].copy()  # Task C needs label
model_df['host_id'] = model_df['host_id'].astype(str)

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(model_df, groups=model_df['host_id']))
train_df = model_df.iloc[train_idx].copy()
temp_df = model_df.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df['host_id']))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print('Train/Val/Test sizes:', len(train_df), len(val_df), len(test_df))
print('Superhost rate train:', train_df['is_superhost'].mean())
print('Superhost rate val:', val_df['is_superhost'].mean())
print('Superhost rate test:', test_df['is_superhost'].mean())

# Verify no host leakage across splits
train_hosts = set(train_df['host_id'])
val_hosts = set(val_df['host_id'])
test_hosts = set(test_df['host_id'])
print('Host overlap train-val:', len(train_hosts & val_hosts))
print('Host overlap train-test:', len(train_hosts & test_hosts))
print('Host overlap val-test:', len(val_hosts & test_hosts))


In [ ]:
# F2 comparison - plain random split vs group split on one simple model
from sklearn.model_selection import train_test_split

# Simple demo features (numeric only, quick)
demo_features = [c for c in ['accommodates','bedrooms','bathrooms_count','minimum_nights','number_of_reviews','review_scores_rating','availability_365','amenities_count'] if c in model_df.columns]
X_demo = model_df[demo_features].fillna(0)
y_demo = model_df['is_superhost'].astype(int)

# Random split (can leak hosts)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_demo, y_demo, test_size=0.2, random_state=RANDOM_STATE, stratify=y_demo)
lr_r = LogisticRegression(max_iter=1000)
lr_r.fit(Xr_tr, yr_tr)
pr = lr_r.predict(Xr_te)
print('Random split accuracy:', accuracy_score(yr_te, pr))
print('Random split F1:', f1_score(yr_te, pr))

# Group split
gss_demo = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_i, te_i = next(gss_demo.split(X_demo, y_demo, groups=model_df['host_id']))
lr_g = LogisticRegression(max_iter=1000)
lr_g.fit(X_demo.iloc[tr_i], y_demo.iloc[tr_i])
pg = lr_g.predict(X_demo.iloc[te_i])
print('Group split accuracy:', accuracy_score(y_demo.iloc[te_i], pg))
print('Group split F1:', f1_score(y_demo.iloc[te_i], pg))


In [ ]:
# F4 - Preprocessing rule
print('Rule: all imputers/scalers/encoders are fit on TRAIN only.')
print('I enforce this by putting preprocessing inside sklearn Pipeline/ColumnTransformer and fitting on train_df (or CV train folds).')
print('I also build any group-stat features from train rows only if I use them.')


## G. Baselines (Checklist G)

- Task C: `DummyClassifier` (majority) with accuracy/precision/recall/F1 on validation.
- Task R: mean/median predictor with RMSE/MAE on validation.
- One plain-language bias/variance sentence tied to your results (update as you compare models in H).

In [ ]:
# Define Task C and Task R feature sets (initial, safe, non-leaky)
# Task C features: listing attributes known at prediction time
task_c_num = [c for c in ['accommodates','bedrooms','bathrooms_count','minimum_nights','number_of_reviews','review_scores_rating','availability_365','amenities_count'] if c in train_df.columns]
task_c_cat = [c for c in ['neighbourhood_cleansed','room_type','property_type'] if c in train_df.columns]

# Task R features: can include similar listing attributes; exclude raw price target
task_r_num = [c for c in ['accommodates','bedrooms','bathrooms_count','minimum_nights','number_of_reviews','review_scores_rating','availability_365','amenities_count','latitude','longitude'] if c in train_df.columns]
task_r_cat = [c for c in ['neighbourhood_cleansed','room_type','property_type'] if c in train_df.columns]

print('Task C num:', task_c_num)
print('Task C cat:', task_c_cat)
print('Task R num:', task_r_num)
print('Task R cat:', task_r_cat)

X_train_c = train_df[task_c_num + task_c_cat].copy()
y_train_c = train_df['is_superhost'].astype(int)
X_val_c = val_df[task_c_num + task_c_cat].copy()
y_val_c = val_df['is_superhost'].astype(int)

# Task R rows need valid price
train_r = train_df[train_df['price_valid']].copy()
val_r = val_df[val_df['price_valid']].copy()
X_train_r = train_r[task_r_num + task_r_cat].copy()
y_train_r = train_r['price_clean'].astype(float)
X_val_r = val_r[task_r_num + task_r_cat].copy()
y_val_r = val_r['price_clean'].astype(float)
print('Task R train/val sizes:', len(train_r), len(val_r))


In [ ]:
# G1 - Task C majority baseline
dummy_c = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy_c.fit(X_train_c, y_train_c)
pred_dummy_c = dummy_c.predict(X_val_c)

baseline_c = {
    'model': 'DummyClassifier(most_frequent)',
    'accuracy': accuracy_score(y_val_c, pred_dummy_c),
    'precision': precision_score(y_val_c, pred_dummy_c, zero_division=0),
    'recall': recall_score(y_val_c, pred_dummy_c, zero_division=0),
    'f1': f1_score(y_val_c, pred_dummy_c, zero_division=0)
}
display(pd.DataFrame([baseline_c]))


In [ ]:
# G2 - Task R mean/median baselines
mean_pred = np.full(len(y_val_r), y_train_r.mean())
median_pred = np.full(len(y_val_r), y_train_r.median())

def reg_metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return {'model': name, 'RMSE_ZAR': rmse, 'MAE_ZAR': mae}

baseline_r = pd.DataFrame([
    reg_metrics(y_val_r, mean_pred, 'Mean predictor'),
    reg_metrics(y_val_r, median_pred, 'Median predictor')
])
display(baseline_r)


## H–J. Modelling, workflow, evaluation (Checklist H, I, J)

Build pipelines (ColumnTransformer for numeric/categorical), try required models, use GroupKFold CV and tuning, handle imbalance, evaluate once on test, and perform error/subgroup analysis.  
Keep the **test set untouched** until final scoring (J1).

In [ ]:
# Shared preprocessors
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess_c = ColumnTransformer([
    ('num', numeric_transformer, task_c_num),
    ('cat', categorical_transformer, task_c_cat)
])
preprocess_r = ColumnTransformer([
    ('num', numeric_transformer, task_r_num),
    ('cat', categorical_transformer, task_r_cat)
])


In [ ]:
# H1 - Logistic regression for Task C
logreg = Pipeline([
    ('preprocess', preprocess_c),
    ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])
logreg.fit(X_train_c, y_train_c)
pred_log = logreg.predict(X_val_c)
proba_log = logreg.predict_proba(X_val_c)[:, 1]

logreg_row = {
    'model': 'LogisticRegression',
    'accuracy': accuracy_score(y_val_c, pred_log),
    'precision': precision_score(y_val_c, pred_log, zero_division=0),
    'recall': recall_score(y_val_c, pred_log, zero_division=0),
    'f1': f1_score(y_val_c, pred_log, zero_division=0),
    'roc_auc': roc_auc_score(y_val_c, proba_log)
}
task_c_results = pd.DataFrame([baseline_c | {'roc_auc': np.nan}, logreg_row])
display(task_c_results)


In [ ]:
# H2 - Top 10 logistic coefficients and C comparison
# Get feature names after preprocessing
ohe = logreg.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
cat_names = ohe.get_feature_names_out(task_c_cat).tolist() if task_c_cat else []
feature_names = task_c_num + cat_names
coefs = logreg.named_steps['model'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefs})
coef_df['abs_coef'] = coef_df['coef'].abs()
display(coef_df.sort_values('abs_coef', ascending=False).head(10))

# Compare C values
c_rows = []
for C in [0.01, 0.1, 1.0, 10.0]:
    m = Pipeline([
        ('preprocess', preprocess_c),
        ('model', LogisticRegression(max_iter=2000, C=C, random_state=RANDOM_STATE))
    ])
    m.fit(X_train_c, y_train_c)
    p = m.predict(X_val_c)
    pr = m.predict_proba(X_val_c)[:,1]
    c_rows.append({'C': C, 'f1': f1_score(y_val_c,p,zero_division=0), 'recall': recall_score(y_val_c,p,zero_division=0), 'roc_auc': roc_auc_score(y_val_c,pr)})
display(pd.DataFrame(c_rows))


In [ ]:
# H3 - Decision tree depth sweep + plot
depth_rows = []
for d in [2,3,4,5,6,8,10,12,None]:
    dt = Pipeline([
        ('preprocess', preprocess_c),
        ('model', DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE))
    ])
    dt.fit(X_train_c, y_train_c)
    tr_pred = dt.predict(X_train_c)
    va_pred = dt.predict(X_val_c)
    depth_rows.append({
        'max_depth': str(d),
        'train_f1': f1_score(y_train_c, tr_pred, zero_division=0),
        'val_f1': f1_score(y_val_c, va_pred, zero_division=0),
        'train_acc': accuracy_score(y_train_c, tr_pred),
        'val_acc': accuracy_score(y_val_c, va_pred)
    })
depth_df = pd.DataFrame(depth_rows)
display(depth_df)

plt.figure(figsize=(8,4))
plt.plot(depth_df['max_depth'].astype(str), depth_df['train_f1'], marker='o', label='train F1')
plt.plot(depth_df['max_depth'].astype(str), depth_df['val_f1'], marker='o', label='val F1')
plt.title('Decision tree: train vs validation F1 by max_depth')
plt.xlabel('max_depth')
plt.ylabel('F1')
plt.legend()
plt.tight_layout()
plt.show()

# Shallow tree for interpretation
shallow_dt = Pipeline([
    ('preprocess', preprocess_c),
    ('model', DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE))
])
shallow_dt.fit(X_train_c, y_train_c)
# Plot tree on transformed feature names is possible but verbose; plot with generic names or top-level.
plt.figure(figsize=(20,8))
plot_tree(shallow_dt.named_steps['model'], max_depth=3, filled=True, fontsize=8, class_names=['Not Superhost','Superhost'])
plt.title('Shallow decision tree (max_depth=3)')
plt.show()


In [ ]:
# H4 - Random forest and gradient boosting
rf = Pipeline([
    ('preprocess', preprocess_c),
    ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
])
gb = Pipeline([
    ('preprocess', preprocess_c),
    ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

models_c = [
    ('DecisionTree', Pipeline([('preprocess', preprocess_c), ('model', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))])),
    ('RandomForest', rf),
    ('GradientBoosting', gb),
]

rows = []
for name, model in models_c:
    model.fit(X_train_c, y_train_c)
    pred = model.predict(X_val_c)
    proba = model.predict_proba(X_val_c)[:,1] if hasattr(model, 'predict_proba') else pred
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_val_c, pred),
        'precision': precision_score(y_val_c, pred, zero_division=0),
        'recall': recall_score(y_val_c, pred, zero_division=0),
        'f1': f1_score(y_val_c, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val_c, proba)
    })

task_c_results = pd.concat([task_c_results, pd.DataFrame(rows)], ignore_index=True)
display(task_c_results)


In [ ]:
# H4b - XGBoost
if XGBOOST_AVAILABLE:
    # scale_pos_weight helps with imbalance; estimate from training labels
    pos = (y_train_c == 1).sum()
    neg = (y_train_c == 0).sum()
    spw = neg / max(pos, 1)
    xgb = Pipeline([
        ('preprocess', preprocess_c),
        ('model', XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            eval_metric='logloss', random_state=RANDOM_STATE,
            scale_pos_weight=spw, n_jobs=-1
        ))
    ])
    xgb.fit(X_train_c, y_train_c)
    pred = xgb.predict(X_val_c)
    proba = xgb.predict_proba(X_val_c)[:,1]
    xgb_row = {
        'model': 'XGBoost',
        'accuracy': accuracy_score(y_val_c, pred),
        'precision': precision_score(y_val_c, pred, zero_division=0),
        'recall': recall_score(y_val_c, pred, zero_division=0),
        'f1': f1_score(y_val_c, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val_c, proba)
    }
    task_c_results = pd.concat([task_c_results, pd.DataFrame([xgb_row])], ignore_index=True)
    display(task_c_results)
else:
    print('Install xgboost to run H4b: pip install xgboost')


In [ ]:
# H5 - Linear regression for Task R
linreg = Pipeline([('preprocess', preprocess_r), ('model', LinearRegression())])
linreg.fit(X_train_r, y_train_r)
pred_lin = linreg.predict(X_val_r)
lin_row = reg_metrics(y_val_r, pred_lin, 'LinearRegression')
task_r_results = pd.concat([baseline_r, pd.DataFrame([lin_row])], ignore_index=True)
display(task_r_results)

# Coefficients (top absolute)
ohe_r = linreg.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
cat_names_r = ohe_r.get_feature_names_out(task_r_cat).tolist() if task_r_cat else []
feat_r = task_r_num + cat_names_r
lr_coefs = pd.DataFrame({'feature': feat_r, 'coef': linreg.named_steps['model'].coef_})
lr_coefs['abs_coef'] = lr_coefs['coef'].abs()
display(lr_coefs.sort_values('abs_coef', ascending=False).head(10))


In [ ]:
# H6 - Ridge and Lasso
ridge = Pipeline([('preprocess', preprocess_r), ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))])
lasso = Pipeline([('preprocess', preprocess_r), ('model', Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000))])

ridge.fit(X_train_r, y_train_r)
lasso.fit(X_train_r, y_train_r)
pred_ridge = ridge.predict(X_val_r)
pred_lasso = lasso.predict(X_val_r)
task_r_results = pd.concat([
    task_r_results,
    pd.DataFrame([reg_metrics(y_val_r, pred_ridge, 'Ridge'), reg_metrics(y_val_r, pred_lasso, 'Lasso')])
], ignore_index=True)
display(task_r_results)

coef_compare = pd.DataFrame({
    'feature': feat_r,
    'linear': linreg.named_steps['model'].coef_,
    'ridge': ridge.named_steps['model'].coef_,
    'lasso': lasso.named_steps['model'].coef_
})
coef_compare['linear_abs'] = coef_compare['linear'].abs()
display(coef_compare.sort_values('linear_abs', ascending=False).head(15))
print('Lasso zero coefficients:', (coef_compare['lasso'] == 0).sum(), '/', len(coef_compare))


In [ ]:
# H7 - Feature importance for best Task C tree/ensemble
# Pick best by F1 among tree/ensemble models present
tree_candidates = task_c_results[task_c_results['model'].isin(['DecisionTree','RandomForest','GradientBoosting','XGBoost'])].copy()
best_tree_name = tree_candidates.sort_values('f1', ascending=False).iloc[0]['model'] if not tree_candidates.empty else None
print('Best tree/ensemble by F1:', best_tree_name)

# Refit chosen model if needed (simple mapping)
best_tree_model = None
if best_tree_name == 'RandomForest':
    best_tree_model = rf
elif best_tree_name == 'GradientBoosting':
    best_tree_model = gb
elif best_tree_name == 'DecisionTree':
    best_tree_model = Pipeline([('preprocess', preprocess_c), ('model', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))]).fit(X_train_c, y_train_c)
elif best_tree_name == 'XGBoost' and XGBOOST_AVAILABLE:
    best_tree_model = xgb

if best_tree_model is not None:
    model_step = best_tree_model.named_steps['model']
    importances = model_step.feature_importances_
    fi = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)
    display(fi)
    plt.figure(figsize=(8,5))
    plt.barh(fi['feature'][::-1], fi['importance'][::-1])
    plt.title(f'Feature importance - {best_tree_name}')
    plt.tight_layout()
    plt.show()

print('Reminder: importance != cause. Importance shows how useful a feature was for splits/predictions in this model, not a causal effect.')


In [ ]:
# H8 - Final Task C comparison and best-fit selection
display(task_c_results.sort_values(['f1','roc_auc'], ascending=False).reset_index(drop=True))

# Decision rubric (write down explicitly)
selection_notes = pd.DataFrame([
    ['Validation metrics', 'F1/recall/ROC-AUC; not accuracy alone'],
    ['Business threshold behavior', 'Can we choose a threshold that fits support capacity?'],
    ['Interpretability', 'Can I explain why it flags a host?'],
    ['Operational cost', 'File size, prediction latency, deployment simplicity'],
    ['Stability', 'CV spread and sensitivity to tuning']
], columns=['Criterion', 'Why it matters'])
display(selection_notes)

print('I select the best-fit model based on this rubric and record the decision in the decision log (L3/O1).')


In [ ]:
# I1 - One Pipeline + ColumnTransformer per task
print('Task C pipeline: preprocess_c + classifier')
print('Task R pipeline: preprocess_r + regressor')
print('The same pipeline object is reused for CV/tuning/saving to avoid preprocessing leakage.')


## K. Unsupervised piece (Checklist K)

K-means on scaled numeric features (elbow + silhouette, interpret centroids), PCA 2D plot colored by cluster with explained variance, and connect clusters to Superhost rate / median price.

In [ ]:
# TODO: clustering and PCA

## L. Save Superhost pipeline for the app (Checklist L)

- Train the **final deployed Superhost pipeline on the 8–12 app features only** (same columns/types the app will send).
- Save the whole pipeline (preprocessing + model) with `joblib`.
- Reload in a fresh cell and verify identical prediction on a sample row. Check file size (<100 MB).

In [ ]:
# TODO: define APP_FEATURES, train final pipeline on them, save/reload/verify
# joblib.dump(superhost_app_pipeline, MODELS_DIR / 'superhost_pipeline.joblib')
# loaded = joblib.load(MODELS_DIR / 'superhost_pipeline.joblib')

## O (ongoing). Decision log and findings

Keep a running decision log (for README, ≥4 key entries): **decision | options considered | why chosen**.  
Examples: leakage columns removed, split strategy, target/price handling, app feature selection, threshold choice, model tradeoffs.

| Decision | Options considered | Why chosen |
|---|---|---|
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |

## Final checks before submission/presentation
- [ ] Kernel restarted and **Run All** completes with no errors/manual edits (O2)
- [ ] Every important result has a written finding in your own words (O3)
- [ ] Test set scored once per task (J1)
- [ ] `models/superhost_pipeline.joblib` loads and matches original predictions (L2)
- [ ] `app.py` runs locally and handles inputs correctly (M2, M4)
- [ ] README includes live URL, credit, 5 findings, scores, limits, run steps, decision log, next steps (O1)
- [ ] You can explain and live-edit random cells (A1 spirit / O5)